# <u><b>Read Outlook Emails</b></u>

In [1]:
import win32com.client

outlook = win32com.client.Dispatch("Outlook.Application")
namespace = outlook.GetNamespace("MAPI")

# RCTI folder
rcti_folder = namespace.Folders["iramanagendra@mmm.com"].Folders["RCTI"]

# Read emails
for mail in rcti_folder.Items:
    print(mail.Subject)

RCTI received from KmartAU
RCTI received from KmartAU
RCTI from OfficeWorksAU


# <b>Get Email body</b>

In [2]:
for mail in rcti_folder.Items:

    has_pdf = any(
        attachment.FileName.lower().endswith(".pdf")
        for attachment in mail.Attachments
    )

    if not has_pdf:

        body = mail.Body

        print("Subject:", mail.Subject)

        # Your regex extraction
        import re

        rcti = re.search(
            r'RCTI Reference Number:\s*([A-Z0-9]+)',
            body
        )

        if rcti:
            print("RCTI Ref No:", rcti.group(1))

Subject: RCTI received from KmartAU
RCTI Ref No: K202608130207
Subject: RCTI received from KmartAU
RCTI Ref No: K202608260206
Subject: RCTI from OfficeWorksAU
RCTI Ref No: 5106172412


In [3]:
import re
import pandas as pd

data = []

items = rcti_folder.Items

for mail in items:

    try:

        # Skip non-mail items
        _ = mail.Subject

        has_pdf = False

        for attachment in mail.Attachments:
            if attachment.FileName.lower().endswith(".pdf"):
                has_pdf = True
                break

        if has_pdf:
            continue

        body = str(mail.Body)

        rcti_ref = re.search(
            r'RCTI Reference Number:\s*([A-Z0-9]+)',
            body
        )

        date_received = re.search(
            r'Date Received:\s*([\d/]+)',
            body
        )

        pattern = r'(\d{2}/\d{2}/\d{4})\s+(\d+)\s+(ORIGINAL ASN)\s+([\d.]+)\s+([\d.]+)\s+(\d+)'

        rows = re.findall(pattern, body)

        for row in rows:

            data.append({
                "Subject": mail.Subject,
                "RCTI_Reference_Number": rcti_ref.group(1) if rcti_ref else "",
                "Date_Received": date_received.group(1) if date_received else "",
                "Receipt_Dt": row[0],
                "ASN_No": row[1],
                "Adjustment": row[2],
                "Total_Amt": float(row[3]),
                "GST_Amt": float(row[4]),
                "POrder_No": str(row[5])
            })

    except Exception as e:
        print(f"Skipped email: {e}")
        continue

df = pd.DataFrame(data)

excel_path = r"C:\Users\AD1HCZZ\3M\ANZ Invoice Reconciliation - ANZ RCTI Email Data\RCTI_Data.xlsx"

df.to_excel(
    excel_path,
    index=False,
    engine="openpyxl"
)

print(df.head())
print(f"\nRows Extracted: {len(df)}")
print(f"Saved: {excel_path}")

                      Subject RCTI_Reference_Number Date_Received  Receipt_Dt  \
0  RCTI received from KmartAU         K202608130207    13/08/2026  12/08/2026   
1  RCTI received from KmartAU         K202608130207    13/08/2026  12/08/2026   
2  RCTI received from KmartAU         K202608130207    13/08/2026  12/08/2026   
3  RCTI received from KmartAU         K202608260206    26/08/2026  25/08/2026   
4     RCTI from OfficeWorksAU            5106172412    26/08/2026  21/08/2026   

       ASN_No    Adjustment  Total_Amt  GST_Amt     POrder_No  
0  6147352944  ORIGINAL ASN   28528.56  2593.51  220060829324  
1  6147371670  ORIGINAL ASN   26410.65  2400.97  220060819251  
2  6147371781  ORIGINAL ASN   68503.85  6227.62  219984949251  
3  6147458254  ORIGINAL ASN   17269.34  1569.94  220246469601  
4  6147475446  ORIGINAL ASN    2549.87   231.80    4506396972  

Rows Extracted: 5
Saved: C:\Users\AD1HCZZ\3M\ANZ Invoice Reconciliation - ANZ RCTI Email Data\RCTI_Data.xlsx


In [7]:
import win32com.client

outlook = win32com.client.Dispatch("Outlook.Application").GetNamespace("MAPI")

for mailbox in outlook.Folders:
    print(mailbox.Name)

Public Folders - iramanagendra@mmm.com
iramanagendra@mmm.com
